# Lab 6: Building ELT Pipelines with PySpark & Delta Lake
## Course: Data Stores & Pipelines (Trimester 2)

In this lab, we shift from ETL to **ELT**. We will load raw transactional data directly into a **Bronze layer** without transformation, and subsequently perform downstream transformations to populate a **Silver layer**.

### Step 1: Initialize SparkSession with Delta Support

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, to_date, upper, lit

spark = SparkSession.builder \
    .appName("Lab6_ELT_Delta_Pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark Session Initialized. Spark Version: {spark.version}")

### Step 2: Extract & Load (EL Phase - Bronze Layer)
In ELT, we extract raw data and immediately **Load** it into our storage layer (Bronze) in its native format with zero modifications.

In [ ]:
# Raw incoming batch simulation
incoming_raw_data = [
    (101, "alice", "electronics", "1500.00", "2026-03-10"),
    (102, "bob", "apparel", " 75.50 ", "2026-03-10"),
    (103, "charlie", "electronics", "INVALID_AMT", "2026-03-11"),
    (104, "david", "home", " 220.00 ", None)
]

columns = ["txn_id", "cust_name", "dept", "raw_amount", "raw_date"]
raw_df = spark.createDataFrame(incoming_raw_data, columns)

# Load directly to Bronze storage path
bronze_path = "./lakehouse/bronze/transactions"
raw_df.write \
    .mode("append") \
    .format("parquet") \
    .save(bronze_path)

print(f"EL Phase Complete: Raw data successfully landed in Bronze layer at {bronze_path}")

### Step 3: Transform Phase (Silver Layer)
Now we read the raw data from the Bronze layer, apply data hygiene (handling bad data types, standardizing strings), and write the conformed data to the **Silver layer**.

In [ ]:
# Read from Bronze
bronze_df = spark.read.parquet(bronze_path)

# Transform: Clean and type-cast
from pyspark.sql.types import DoubleType

silver_df = bronze_df \
    .withColumn("amount", col("raw_amount").cast(DoubleType())) \
    .withColumn("txn_date", to_date(col("raw_date"), "yyyy-MM-dd")) \
    .withColumn("cust_name", upper(col("cust_name"))) \
    .withColumn("dept", upper(col("dept"))) \
    .withColumn("ingested_at", current_timestamp()) \
    .select("txn_id", "cust_name", "dept", "amount", "txn_date", "ingested_at")

# Load to Silver storage path
silver_path = "./lakehouse/silver/transactions"
silver_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(silver_path)

print("--- Silver Layer Transformed Data ---")
silver_df.show(truncate=False)